# Importar librerias y setup

In [10]:
import os, json, glob, random
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from tqdm import tqdm

from skimage import measure
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from collections import defaultdict


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


# SegFormer

In [3]:
import torch
from transformers import SegformerImageProcessor, SegformerModel, SegformerForSemanticSegmentation

processor_seg = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
segformer = SegformerModel.from_pretrained("mattmdjaga/segformer_b2_clothes").to(device)
segformer.eval()

SegformerModel(
  (encoder): SegformerEncoder(
    (patch_embeddings): ModuleList(
      (0): SegformerOverlapPatchEmbeddings(
        (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
        (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      )
      (1): SegformerOverlapPatchEmbeddings(
        (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
      (2): SegformerOverlapPatchEmbeddings(
        (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
      )
      (3): SegformerOverlapPatchEmbeddings(
        (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      )
    )
    (block): ModuleList(
      (0): ModuleList(
        (0): Segform

# Dino v3

In [4]:
#!git clone https://github.com/facebookresearch/dinov3.git
#!gdown --id 1Fznrc_pDwp7iaUBhAoWUKPI1vsGpHy8m

In [5]:
import torch
from torchvision import transforms

dinov3 = torch.hub.load("dinov3", 'dinov3_vitb16', source='local', weights="/content/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth")
dinov3 = dinov3.to(device)

# Carga del dataset

In [6]:
DIR_MODA = Path("modanet")

DIR_IMAGENES = DIR_MODA / "images"
DIR_ANOTACIONES = DIR_MODA / "annotations-seg"
DIR_PRED_DINO = DIR_MODA / "pred_dino"
DIR_PRED_SEGFORMER = DIR_MODA / "pred_segformer"


'''
!mkdir {DIR_MODA}
!wget "https://www.dropbox.com/scl/fo/g0or9rc44z7ey8oj8b5gv/AJnHmMLpYvYNYj-DK8Vf5oA/annotations-seg.tar?rlkey=l61rkqa63s9kv57bh1tp1oiie&dl=0" -q -O annotations-seg.tar
!wget "https://www.dropbox.com/scl/fo/g0or9rc44z7ey8oj8b5gv/AMvjsHNsh406QE1R7fhEhG8/images_modanet.tar?rlkey=l61rkqa63s9kv57bh1tp1oiie&dl=0" -q -O images_modanet.tar

!tar -xf annotations-seg.tar -C {DIR_MODA}
!tar -xf images_modanet.tar -C {DIR_MODA}
'''

all_files = list(DIR_ANOTACIONES.glob("*.json"))
json_files = random.sample(all_files, 10)

modanet_id2label = {1:"bag", 2:"belt", 3:"boots", 4:"footwear", 5:"outer",
                   6:"dress", 7:"sunglasses", 8:"pants", 9:"top", 10:"shorts",
                   11:"skirt", 12:"headwear", 13:"scarf & tie"}





In [7]:
transform_seg = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_dino = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


image = Image.open(DIR_IMAGENES/ "1000033.jpg")


image = image.convert("RGB")
inputs_seg = processor_seg(images=image, return_tensors="pt").to(device)

img_tensor = transform_seg(image).unsqueeze(0).to(device)

with torch.no_grad():
    outputs_seg = segformer(**inputs_seg)
    outputs_dino = dinov3(img_tensor, is_training=True)['x_norm_patchtokens']